# Del papel a la base de datos, a escala

**La idea.** Una ficha se lee en el chat. Cien no: nadie sube cien documentos, uno por uno, a Gemini. Para escalar, **la ficha de campos se vuelve código**: una función que lee una imagen, un ciclo que recorre todas, un JSON por documento, una tabla al final y una verificación.

**Los documentos.** 8 fichas técnicas **simuladas**, con el formato del archivo de emisoras del Ministerio de Comunicaciones. Como las armamos nosotros, **sabemos la verdad**, y el cuaderno mide cuánto acierta la IA. Cada ficha esconde una trampa: un número desteñido, un tachón, un renglón cortado, una palabra mal escrita en el original.

**Lo que necesita.** Una llave gratis de la API de Gemini (paso 0). Si no tiene, use el plan B: `USAR_EJEMPLO = True` en el paso 2 carga las salidas de una corrida de ejemplo.

## Usted no tiene que programar esto: tiene que saber pedirlo

Este cuaderno es lo que escribiría alguien que programa. Usted no necesita aprender a escribirlo: necesita **pedírselo bien a la IA**, en Antigravity, en el panel de Gemini de Colab o en cualquier asistente que escriba código. Este es el prompt que produce un código como el de abajo:

```text
Tengo una carpeta con imágenes escaneadas de fichas técnicas. Escribe un programa en Python que:
1. Recorra todas las imágenes de la carpeta.
2. A cada imagen le aplique mi ficha de campos (te la pego abajo) usando la API de Gemini. La llave se lee de un secreto llamado GOOGLE_API_KEY y nunca se escribe en el código.
3. Guarde un archivo JSON por imagen y, si se interrumpe, siga donde iba sin repetir las que ya hizo.
4. Respete el límite de la cuenta gratis: una pausa entre consultas y reintentos si la API dice que espere.
5. Arme una tabla CSV con todos los campos de todas las fichas.
6. Saque una muestra al azar de 10 celdas para que yo las revise a mano contra las imágenes.
7. Escriba un registro de auditoría con la fecha, el modelo, el número de documentos y los errores.
Antes de correrlo, explícame el código paso a paso.

MI FICHA DE CAMPOS:
[pegue aquí su ficha de campos]
```

**El criterio sigue siendo suyo:** la ficha de campos, la regla de ILEGIBLE antes que adivinar, la muestra de 10 celdas y el registro. La IA solo escribe el código que los aplica a cien documentos.

## Paso 0 · Su llave, guardada como secreto (2 minutos)

1. Entre a https://aistudio.google.com/apikey con su cuenta de Google y cree una llave (es gratis).
2. En Colab, a la izquierda, abra el ícono de la llave 🔑 (**Secretos**).
3. **Agregar nuevo secreto** → Nombre: `GOOGLE_API_KEY` → Valor: su llave → active **Acceso del cuaderno**.

**Nunca pegue la llave en el código:** el cuaderno la lee del secreto. Así puede compartir el cuaderno sin regalar su llave.

In [ ]:
import os, json, time, base64, random, requests
import pandas as pd
from pathlib import Path

BASE = "https://alonsorh-econ.github.io/diplomado-ia-investigacion/sesion-08/ocr_escala/"
ARCHIVOS = [f"F0{i}.png" for i in range(1, 9)]
Path("fichas").mkdir(exist_ok=True)
Path("salidas").mkdir(exist_ok=True)
for a in ARCHIVOS:
    r = requests.get(BASE + "fichas/" + a, timeout=60)
    r.raise_for_status()
    Path("fichas", a).write_bytes(r.content)
verdad = pd.read_csv(BASE + "verdad.csv")
print(f"{len(ARCHIVOS)} fichas descargadas en la carpeta fichas/")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, ejes = plt.subplots(2, 4, figsize=(16, 9))
for eje, a in zip(ejes.flat, ARCHIVOS):
    eje.imshow(Image.open(Path("fichas", a)))
    eje.set_title(a)
    eje.axis("off")
plt.tight_layout()
plt.show()

## Paso 1 · La ficha de campos

Es el mismo criterio que usamos en el chat, ahora como texto dentro del código. Si cambia de documento, cambia esto.

In [ ]:
PROMPT = """Eres un transcriptor de archivo histórico. Te paso la imagen de una ficha técnica de una emisora. Conviértela en UN objeto JSON con estas claves:
distintivo, ciudad, frecuencia, potencia, marca, antena, altura, fecha, propietario, folio.
Para cada clave devuelve un objeto {"valor": ..., "tachado": ..., "confianza": "alta" | "media" | "baja"}.
REGLAS
1. Copia tal cual, sin corregir ortografía ni cambiar unidades ni separadores (si dice «1.230 Kcs/S», escribe «1.230 Kcs/S»).
2. Si no puedes leer algo, el valor es "ILEGIBLE". Si el renglón está cortado, el valor es "ILEGIBLE". Nunca adivines.
3. Si algo está tachado, pon en "tachado" lo que alcances a leer de lo tachado (o "ILEGIBLE") y en "valor" solo lo vigente.
4. "folio" es el número escrito a mano en la esquina superior.
5. Ignora la línea final que dice que la ficha es simulada.
Responde solo con el JSON."""

## Paso 2 · Una función que lee una ficha

La función manda una imagen y el PROMPT a la API de Gemini, pide la respuesta en JSON y, si la cuenta gratis llega a su límite, espera y reintenta.

In [ ]:
MODELO = "gemini-3.5-flash"   # modelo de la API; puede cambiarlo
USAR_EJEMPLO = False          # plan B: póngalo en True si no tiene llave


def llave():
    from google.colab import userdata
    return userdata.get("GOOGLE_API_KEY")


def leer_ficha(ruta, intentos=3):
    """Envía una imagen y el PROMPT a Gemini y devuelve el JSON como diccionario."""
    img = base64.b64encode(Path(ruta).read_bytes()).decode()
    cuerpo = {"contents": [{"role": "user", "parts": [
                 {"inline_data": {"mime_type": "image/png", "data": img}},
                 {"text": PROMPT}]}],
              "generationConfig": {"responseMimeType": "application/json", "temperature": 0}}
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO}:generateContent"
    for k in range(intentos):
        r = requests.post(url, json=cuerpo, headers={"x-goog-api-key": llave()}, timeout=120)
        if r.status_code == 200:
            partes = r.json()["candidates"][0]["content"]["parts"]
            texto = "".join(p.get("text", "") for p in partes if not p.get("thought"))
            return json.loads(texto)
        if r.status_code in (429, 500, 503):   # límite de la cuenta gratis o servidor ocupado
            espera = 15 * (k + 1)
            print(f"   {r.status_code}: espero {espera} s y reintento")
            time.sleep(espera)
            continue
        raise RuntimeError(f"Error {r.status_code}: {r.text[:300]}")
    raise RuntimeError("No respondió después de varios intentos")

## Paso 3 · El ciclo: todas las fichas, un JSON por ficha

Si se interrumpe, vuelva a correr la celda: sigue donde iba y no repite las que ya están en `salidas/`.

In [ ]:
if USAR_EJEMPLO:
    for a in ARCHIVOS:
        nombre = a.replace(".png", ".json")
        r = requests.get(BASE + "salidas_ejemplo/" + nombre, timeout=60)
        Path("salidas", nombre).write_text(r.text, encoding="utf-8")
    print("Plan B: se cargaron las salidas de una corrida de ejemplo.")
else:
    t0 = time.time()
    for a in ARCHIVOS:
        destino = Path("salidas", a.replace(".png", ".json"))
        if destino.exists():
            print(f"{a}: ya estaba")
            continue
        datos = leer_ficha(Path("fichas", a))
        destino.write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"{a}: listo · {time.time() - t0:.0f} s")
        time.sleep(4)   # respeta el límite de consultas por minuto de la cuenta gratis
print("JSON en salidas/:", sorted(p.name for p in Path("salidas").glob("*.json")))

## Paso 4 · De los JSON a una tabla

In [ ]:
CAMPOS = ["distintivo", "ciudad", "frecuencia", "potencia", "marca", "antena", "altura", "fecha", "propietario", "folio"]
filas = []
for a in ARCHIVOS:
    j = json.loads(Path("salidas", a.replace(".png", ".json")).read_text(encoding="utf-8"))
    fila = {"archivo": a}
    for c in CAMPOS:
        v = j.get(c, {})
        fila[c] = v.get("valor") if isinstance(v, dict) else v
        fila[c + "_confianza"] = v.get("confianza") if isinstance(v, dict) else None
    filas.append(fila)
tabla = pd.DataFrame(filas)
tabla.to_csv("fichas_extraidas.csv", index=False)
display(tabla[["archivo"] + CAMPOS])

## Paso 5 · La verificación: ¿cuánto acertó?

Aquí se puede medir porque conocemos la verdad. Con documentos reales no la tiene: por eso, en el paso 6, se revisa a mano una muestra.

In [ ]:
import unicodedata, re


def norm(x):
    x = "" if x is None else str(x)
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", x).strip(" .")


v = verdad.set_index("archivo")
errores, aciertos, total = [], 0, 0
for f in filas:
    for c in CAMPOS:
        esperado, leido = v.loc[f["archivo"], c], f[c]
        total += 1
        if norm(esperado) == norm(leido):
            aciertos += 1
        else:
            errores.append({"archivo": f["archivo"], "campo": c, "dice la ficha": esperado,
                            "leyó la IA": leido, "confianza": f[c + "_confianza"]})
print(f"Acierto: {aciertos} de {total} celdas ({100 * aciertos / total:.0f} %)")
display(pd.DataFrame(errores))
print("\nLa trampa de cada ficha:")
for a, t in v["trampa"].items():
    print(f"  {a}: {t}")

## Paso 6 · Con documentos reales: la muestra que se revisa a mano

In [ ]:
muestra = random.Random(2026).sample([(f["archivo"], c, f[c]) for f in filas for c in CAMPOS], 10)
print("Revise estas 10 celdas contra la imagen original y anote cuántas acertó:\n")
for a, c, val in muestra:
    print(f"  {a} · {c}: {val}")

## Paso 7 · El registro de auditoría

In [ ]:
import datetime as dt

registro = {"fecha": dt.datetime.now().isoformat(timespec="minutes"),
            "modelo": "ejemplo" if USAR_EJEMPLO else MODELO,
            "documentos": len(ARCHIVOS), "celdas": total, "aciertos": aciertos, "errores": len(errores),
            "regla": "ILEGIBLE antes que adivinar; 10 celdas revisadas a mano por lote"}
pd.DataFrame([registro]).to_csv("auditoria_ocr.csv", mode="a", index=False,
                                header=not Path("auditoria_ocr.csv").exists())
print(registro)

## Para llevarlo a su proyecto

- Cambie las imágenes por las suyas (súbalas a la carpeta `fichas/` desde el panel de archivos de la izquierda) y la lista `ARCHIVOS`.
- Cambie el `PROMPT` por su ficha de campos.
- Para cien documentos el código es el mismo: solo cambia la lista de archivos. La pausa de 4 segundos respeta el límite de la cuenta gratis.
- Descargue `fichas_extraidas.csv` y `auditoria_ocr.csv` y guárdelos en su `03_fuentes`. Sin verdad conocida, la tasa de acierto sale de revisar a mano la muestra del paso 6.